In [0]:
spark

In [0]:
df = spark.read.csv(
    "s3://superstore-data-eng-sg-2026/raw/",
    header=True,
    inferSchema=True
)

df.show(5)

+--------------+---------+-------------+---------------+----------+-----------+------+---------------+------------+--------+--------+--------+--------+
|     Ship Mode|  Segment|      Country|           City|     State|Postal Code|Region|       Category|Sub-Category|   Sales|Quantity|Discount|  Profit|
+--------------+---------+-------------+---------------+----------+-----------+------+---------------+------------+--------+--------+--------+--------+
|  Second Class| Consumer|United States|      Henderson|  Kentucky|      42420| South|      Furniture|   Bookcases|  261.96|       2|     0.0| 41.9136|
|  Second Class| Consumer|United States|      Henderson|  Kentucky|      42420| South|      Furniture|      Chairs|  731.94|       3|     0.0| 219.582|
|  Second Class|Corporate|United States|    Los Angeles|California|      90036|  West|Office Supplies|      Labels|   14.62|       2|     0.0|  6.8714|
|Standard Class| Consumer|United States|Fort Lauderdale|   Florida|      33311| South|  

In [0]:
print("Rows:", df.count())
print("Columns:", len(df.columns))

df.printSchema()

Rows: 10014
Columns: 13
root
 |-- Ship Mode: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [0]:
from pyspark.sql.functions import col,when,count

df1 = df.select([
    count(when(col(i).isNull(),i)).alias(i)
    for i in df.columns
])
display(df1)

Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
print("Total Rows:", df.count())
print("Distinct Rows:", df.distinct().count())
df = df.dropDuplicates()
print("Distinct Rows after dropping duplicates:", df.count())

Total Rows: 10014
Distinct Rows: 9997
Distinct Rows after dropping duplicates: 9997


In [0]:

display(df.describe())
display(df.filter(col("Profit")<0).count())

summary,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
count,9997,9997,9997,9997,9997,9997,9997,9997,9997,9997,9997,9997,9997
mean,null,null,null,null,null,55153.21636490947,null,null,null,230.94768933679742,3.78963689106732,0.15606581974593364,28.83933406021785
stddev,null,null,null,null,null,32056.81753797447,null,null,null,624.2033934040444,2.2285976077143905,0.20632747097556353,234.3488847676628
min,First Class,Consumer,United States,Aberdeen,Alabama,1040,Central,Furniture,Accessories,0.444,1,0.0,-6599.978
max,Standard Class,Home Office,United States,Yuma,Wyoming,99301,West,Technology,Tables,22638.48,14,0.8,8399.976


1871

In [0]:
from pyspark.sql.functions import *
display(df.groupBy(col("Category")).agg(sum("Sales").alias("Total_Sales")).orderBy(col("Total_Sales"), ascending = False))

Category,Total_Sales
Technology,845754.0029999963
Furniture,743976.3132999989
Office Supplies,719053.7340000019


In [0]:
df.groupBy("State").agg(sum(col("profit")).alias("Total_Profit")).orderBy(col("Total_Profit").desc()).show(5)

+----------+-----------------+
|     State|     Total_Profit|
+----------+-----------------+
|California|76375.78910000011|
|  New York|74945.46220000004|
|Washington|33423.23750000002|
|  Michigan|24368.09029999999|
|  Virginia|18597.95039999999|
+----------+-----------------+
only showing top 5 rows


In [0]:
df.groupBy("Sub-Category").agg(sum(col("Quantity")).alias("Total_Quantity")).orderBy(col("Total_Quantity").desc()).show(5)

+------------+--------------+
|Sub-Category|Total_Quantity|
+------------+--------------+
|     Binders|          5982|
|       Paper|          5166|
| Furnishings|          3560|
|      Phones|          3292|
|     Storage|          3167|
+------------+--------------+
only showing top 5 rows


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
state_profit = df.groupBy("State","Country","Postal Code","Category").agg(sum(col("profit")).alias("Total_profit")).orderBy(col("Total_profit"))
window_part = Window.orderBy(col("Total_profit").desc())
ranked_data = state_profit.withColumn("Rank",rank().over(window_part))
ranked_data.show(5)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+-------------+-----------+---------------+------------------+----+
|     State|      Country|Postal Code|       Category|      Total_profit|Rank|
+----------+-------------+-----------+---------------+------------------+----+
|  New York|United States|      10024|     Technology|18647.745100000004|   1|
|Washington|United States|      98115|     Technology| 9031.912099999998|   2|
|   Indiana|United States|      47905|     Technology|          8399.976|   3|
|  New York|United States|      10035|     Technology| 8154.912499999999|   4|
|  New York|United States|      10009|Office Supplies| 6907.775100000002|   5|
+----------+-------------+-----------+---------------+------------------+----+
only showing top 5 rows


In [0]:
grouped_data = df.groupBy("State","City",).agg(round(sum(col("Sales")),2).alias("Top_Sales"))
window_part = Window.partitionBy("State").orderBy(col("Top_Sales").desc())
ranked_data = grouped_data.withColumn("Rank",rank().over(window_part))
display(ranked_data.filter(col("Rank") <= 2))


State,City,Top_Sales,Rank
Alabama,Mobile,5462.99,1
Alabama,Montgomery,3722.73,2
Arizona,Phoenix,12250.26,1
Arizona,Tucson,6313.02,2
Arkansas,Fayetteville,3742.81,1
Arkansas,Little Rock,3560.35,2
California,Los Angeles,175831.9,1
California,San Francisco,112577.17,2
Colorado,Denver,12538.79,1
Colorado,Louisville,5070.42,2


In [0]:
table1 = df.groupBy("State").agg(
    max(col("Profit")).alias("Max_Profit")
)

window_part = Window.partitionBy("State").orderBy(col("Max_Profit").desc())

ranked_data = table1.withColumn(
    "Rank",
    rank().over(window_part)
)

display(
    ranked_data.filter(col("Rank") <= 1)
               .select("State","Max_Profit")
)

State,Max_Profit
Alabama,1459.2
Arizona,310.0
Arkansas,843.1706
California,1906.485
Colorado,247.996
Connecticut,294.671
Delaware,5039.9856
District of Columbia,648.5624
Florida,327.5922
Georgia,3177.475


In [0]:
from pyspark.sql.functions import *
df = df.withColumn("Profit_or_loss",
              when(col("Profit") > 0, "Profit")
              .when(col("Profit") < 0, "Loss")
              .otherwise("No Profit or Loss"))
display(df.head(100))

Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit,Profit_or_loss
Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.94,3,0.0,219.582,Profit
Standard Class,Consumer,United States,Los Angeles,California,90032,West,Technology,Phones,907.152,6,0.2,90.7152,Profit
Second Class,Consumer,United States,San Francisco,California,94109,West,Office Supplies,Binders,22.72,4,0.2,7.384,Profit
Standard Class,Consumer,United States,Philadelphia,Pennsylvania,19140,East,Office Supplies,Art,15.76,2,0.2,3.546,Profit
First Class,Corporate,United States,Houston,Texas,77041,Central,Office Supplies,Storage,27.24,3,0.2,2.724,Profit
First Class,Corporate,United States,San Jose,California,95123,West,Office Supplies,Storage,80.88,6,0.0,21.0288,Profit
Standard Class,Consumer,United States,San Antonio,Texas,78207,Central,Technology,Machines,8159.952,8,0.4,-1359.992,Loss
Second Class,Consumer,United States,Newark,Ohio,43055,East,Furniture,Chairs,396.802,7,0.3,-11.3372,Loss
Second Class,Home Office,United States,Monroe,Louisiana,71203,South,Technology,Phones,149.95,5,0.0,41.986,Profit
Second Class,Home Office,United States,Monroe,Louisiana,71203,South,Technology,Accessories,29.0,2,0.0,7.25,Profit


In [0]:
df = df.withColumn(
    "Profit_Percentage",
    round((col("Profit") / col("Sales")) * 100, 2)
)

display(df.head(10))


Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit,Profit_or_loss,Profit_Percentage
Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.94,3,0.0,219.582,Profit,30.0
Standard Class,Consumer,United States,Los Angeles,California,90032,West,Technology,Phones,907.152,6,0.2,90.7152,Profit,10.0
Second Class,Consumer,United States,San Francisco,California,94109,West,Office Supplies,Binders,22.72,4,0.2,7.384,Profit,32.5
Standard Class,Consumer,United States,Philadelphia,Pennsylvania,19140,East,Office Supplies,Art,15.76,2,0.2,3.546,Profit,22.5
First Class,Corporate,United States,Houston,Texas,77041,Central,Office Supplies,Storage,27.24,3,0.2,2.724,Profit,10.0
First Class,Corporate,United States,San Jose,California,95123,West,Office Supplies,Storage,80.88,6,0.0,21.0288,Profit,26.0
Standard Class,Consumer,United States,San Antonio,Texas,78207,Central,Technology,Machines,8159.952,8,0.4,-1359.992,Loss,-16.67
Second Class,Consumer,United States,Newark,Ohio,43055,East,Furniture,Chairs,396.802,7,0.3,-11.3372,Loss,-2.86
Second Class,Home Office,United States,Monroe,Louisiana,71203,South,Technology,Phones,149.95,5,0.0,41.986,Profit,28.0
Second Class,Home Office,United States,Monroe,Louisiana,71203,South,Technology,Accessories,29.0,2,0.0,7.25,Profit,25.0


In [0]:
from pyspark.sql.functions import current_timestamp

df = df.withColumn(
    "load_timestamp",
    current_timestamp()
)

display(df.select("load_timestamp"))

load_timestamp
2026-06-09T18:34:59.617Z
2026-06-09T18:34:59.617Z
2026-06-09T18:34:59.617Z
2026-06-09T18:34:59.617Z
2026-06-09T18:34:59.617Z
2026-06-09T18:34:59.617Z
2026-06-09T18:34:59.617Z
2026-06-09T18:34:59.617Z
2026-06-09T18:34:59.617Z
2026-06-09T18:34:59.617Z


In [0]:
df.printSchema()

root
 |-- Ship Mode: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Profit_or_loss: string (nullable = false)
 |-- Profit_Percentage: double (nullable = true)
 |-- load_timestamp: timestamp (nullable = false)



In [0]:
df.write.mode("overwrite").parquet(
    "s3://superstore-data-eng-sg-2026/processed/superstore_parquet/"
)

In [0]:
df_parquet = spark.read.parquet(
    "s3://superstore-data-eng-sg-2026/processed/superstore_parquet/"
)
display(df_parquet.limit(10))
df_parquet.printSchema()
df_parquet.count()



Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit,Profit_or_loss,Profit_Percentage,load_timestamp
Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.94,3,0.0,219.582,Profit,30.0,2026-06-09T18:35:10.658Z
Standard Class,Consumer,United States,Los Angeles,California,90032,West,Technology,Phones,907.152,6,0.2,90.7152,Profit,10.0,2026-06-09T18:35:10.658Z
Second Class,Consumer,United States,San Francisco,California,94109,West,Office Supplies,Binders,22.72,4,0.2,7.384,Profit,32.5,2026-06-09T18:35:10.658Z
Standard Class,Consumer,United States,Philadelphia,Pennsylvania,19140,East,Office Supplies,Art,15.76,2,0.2,3.546,Profit,22.5,2026-06-09T18:35:10.658Z
First Class,Corporate,United States,Houston,Texas,77041,Central,Office Supplies,Storage,27.24,3,0.2,2.724,Profit,10.0,2026-06-09T18:35:10.658Z
First Class,Corporate,United States,San Jose,California,95123,West,Office Supplies,Storage,80.88,6,0.0,21.0288,Profit,26.0,2026-06-09T18:35:10.658Z
Standard Class,Consumer,United States,San Antonio,Texas,78207,Central,Technology,Machines,8159.952,8,0.4,-1359.992,Loss,-16.67,2026-06-09T18:35:10.658Z
Second Class,Consumer,United States,Newark,Ohio,43055,East,Furniture,Chairs,396.802,7,0.3,-11.3372,Loss,-2.86,2026-06-09T18:35:10.658Z
Second Class,Home Office,United States,Monroe,Louisiana,71203,South,Technology,Phones,149.95,5,0.0,41.986,Profit,28.0,2026-06-09T18:35:10.658Z
Second Class,Home Office,United States,Monroe,Louisiana,71203,South,Technology,Accessories,29.0,2,0.0,7.25,Profit,25.0,2026-06-09T18:35:10.658Z


root
 |-- Ship Mode: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Profit_or_loss: string (nullable = true)
 |-- Profit_Percentage: double (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)



9997

In [0]:
df_parquet.count()

9997

### Delta Part


In [0]:
for c in df_parquet.columns:
    df_parquet = df_parquet.withColumnRenamed(
        c,
        c.replace(" ", "_")
    )

In [0]:
df_parquet.printSchema()

root
 |-- Ship_Mode: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Profit_or_loss: string (nullable = true)
 |-- Profit_Percentage: double (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)



In [0]:
df_parquet.write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3://superstore-data-eng-sg-2026/processed/superstore_delta/")

In [0]:
df_delta = spark.read \
    .format("delta") \
    .load("s3://superstore-data-eng-sg-2026/processed/superstore_delta/")

display(df_delta.limit(5))

Ship_Mode,Segment,Country,City,State,Postal_Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit,Profit_or_loss,Profit_Percentage,load_timestamp
Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.94,3,0.0,219.582,Profit,30.0,2026-06-09T17:37:54.247Z
Standard Class,Consumer,United States,Los Angeles,California,90032,West,Technology,Phones,907.152,6,0.2,90.7152,Profit,10.0,2026-06-09T17:37:54.247Z
Second Class,Consumer,United States,San Francisco,California,94109,West,Office Supplies,Binders,22.72,4,0.2,7.384,Profit,32.5,2026-06-09T17:37:54.247Z
Standard Class,Consumer,United States,Philadelphia,Pennsylvania,19140,East,Office Supplies,Art,15.76,2,0.2,3.546,Profit,22.5,2026-06-09T17:37:54.247Z
First Class,Corporate,United States,Houston,Texas,77041,Central,Office Supplies,Storage,27.24,3,0.2,2.724,Profit,10.0,2026-06-09T17:37:54.247Z
